# VAK Simulated Dataset Exploration

Material Engagement - Beta (simulated dataset).

Exploratory analysis and validation of `data/simulated_dataset.csv`, used to
train the VAK classifier (Visual / Auditory / Kinesthetic): class balance,
descriptive statistics and feature distributions.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from preprocess import NUMERIC_FEATURES, TARGET_COLUMN

sns.set_theme(style='whitegrid')
DATA_PATH = os.path.join('..', 'data', 'simulated_dataset.csv')
REPORTS_DIR = os.path.join('..', 'reports')

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

## 1. Data types and missing values

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else 'No missing values')

## 2. Class balance

Distribution of the VAK label. The dataset is somewhat imbalanced; `train.py`
applies SMOTE on the training set to correct it.

In [ ]:
counts = df[TARGET_COLUMN].value_counts()
print(counts)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN, order=counts.index, hue=TARGET_COLUMN, palette='viridis', legend=False)
plt.title('VAK class distribution')
plt.xlabel('Learning style')
plt.ylabel('Number of students')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'class_distribution.png'), dpi=120)
plt.show()

## 3. Descriptive statistics of numeric features

In [ ]:
df[NUMERIC_FEATURES].describe().T

## 4. VAK scores by actual style

Each style is expected to show its matching score as the highest.

In [ ]:
score_cols = ['visual_score', 'auditory_score', 'kinesthetic_score']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, score_cols):
    sns.boxplot(data=df, x=TARGET_COLUMN, y=col, hue=TARGET_COLUMN, palette='Set2', ax=ax, legend=False)
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('VAK scores by actual style')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'vak_scores_by_class.png'), dpi=120)
plt.show()

## 5. Correlation between numeric features

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[NUMERIC_FEATURES].corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', square=True)
plt.title('Correlation matrix - numeric features')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'correlation_matrix.png'), dpi=120)
plt.show()

## Conclusions

- The dataset holds the 7 numeric features plus the VAK label.
- Classes are imbalanced; `train.py` applies SMOTE on the training set.
- The `visual_score`, `auditory_score` and `kinesthetic_score` features are the
  most discriminative, as the model's feature importance confirms.
- No relevant missing values; the pipeline imputes with the median as a safeguard.